Day 1 (chapters 1 & 2 from Spark:The definitive guide)

1. What is bigdata?
---- Big Data refers to extremely large and complex datasets that cannot be easily managed, processed, or analyzed using traditional data    processing tools. These datasets grow continuously and come from various sources such as social media, IoT devices, financial transactions, and more.

2. Why spark?
---- Apache Spark is powerful in both data manipulation and cluster management. Spark do managing and coordinating the execution of tasks on data across a cluster of computers because one single machine cannot be able to complete the data processing but a group of machines can do it.

3. What is spark?
---- Apache Spark is a unified computing engine and a set of libraries for parallel data processing on computer clusters. It is widely used for big data analytics, machine learning, real-time data streaming, and graph processing.
Spark's toolkit:
a. structured streaming
b. advanced analytics
c. libraries and ecosystem
d. structured API's -- datasets, dataframes, SQL
e. low-level API's -- RDD's, distributed variables

4. Internals of spark?
---- Spark follows a master-slave architecture with the following components:
A. Driver Program (Master Node) ---- Controls the execution of the Spark application. Creates the SparkContext, which is the entry point for interacting with Spark. Converts transformations and actions into a DAG (Directed Acyclic Graph).
Schedules tasks to be executed on worker nodes.
B. Cluster Manager ---- Allocates resources to Spark applications. Can use different cluster managers: Standalone Mode – Default Spark cluster manager. YARN (Hadoop) – Runs Spark on Hadoop clusters. Mesos – A general-purpose cluster manager. Kubernetes – Manages Spark in containerized environments.
C. Executor (Worker Node) ---- Runs on each worker node and executes tasks. Each executor has its own JVM process and memory space. Stores computed data in memory or disk for reuse (RDDs, DataFrames).

5. Highlevel API of spark? Sparksession, Dataframe, Partitions, Transformation, Actions, Lazy Evaluation
---- Apache Spark provides high-level APIs for efficient data processing. The key components include SparkSession, DataFrame, Partitions, Transformations, Actions, and Lazy Evaluation.
a. SparkSession is the entry point for any Spark application. It allows us to interact with DataFrames, Datasets, and SQL APIs.
b. DataFrame is a distributed collection of structured data (like a SQL table).
c. Partitions allow Spark to process data in parallel.
d. Transformations don’t execute immediately but define a logical plan which is used when we are trying to excecute code.
e. Actions trigger computation and return results.
f. Lazy evaulation means that Spark will wait until the very last moment to execute the graph of computation instructions. In Spark, instead of modifying the data immediately when you express some operation, you build up a plan of transformations that you would like to apply to your source data. By waiting until the last minute to execute the code.

Day 2 (chapter 4, 5)

1. Structured API - Dataframes, SQL and Dataset
---- Apache Spark’s Structured API consists of DataFrames, SQL, and Datasets.
a. DataFrame is a distributed collection of structured data, organized in rows and columns (similar to a SQL table). It is the most widely used API in Spark.
b. Spark provides an interface to run SQL queries directly on DataFrames.
c. Dataset is a strongly-typed, distributed collection of objects. It is available in Scala and Java (not in Python).

2. Basic Structured Operation
        1. Schemas
        2. Columns and Expressions
        3. Records and Rows
        4. Create Rows
        5. Dataframe Transformation
        6. Creating Dataframes
        7. select and selectExpr
        8. Adding columns
        9. Renaming columns
        10. Changing column's type
        11. Filtering rows
        12. Getting unique rows
        13. Sorting
        14. Limit
        15. Repartition and Coalesce

In [1]:
import pyspark

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark assignment Notebook").getOrCreate()

In [3]:
spark

In [5]:
df = spark.read.format("json").load("2015-summary.json")

In [6]:
# Schemas
df.printSchema()

root
 |-- DEST_COUNTRY_NAME: string (nullable = true)
 |-- ORIGIN_COUNTRY_NAME: string (nullable = true)
 |-- count: long (nullable = true)



In [7]:
# columns
df.columns

['DEST_COUNTRY_NAME', 'ORIGIN_COUNTRY_NAME', 'count']

In [8]:
# expressions
from pyspark.sql.functions import col, column

df.select(col("count"))
df.select(column("count"))

DataFrame[count: bigint]

In [9]:
# records and rows
df.show(5)

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Romania|   15|
|    United States|            Croatia|    1|
|    United States|            Ireland|  344|
|            Egypt|      United States|   15|
|    United States|              India|   62|
+-----------------+-------------------+-----+
only showing top 5 rows



In [10]:
# create rows
from pyspark.sql import Row
df = df.union(spark.createDataFrame([Row(DEST_COUNTRY_NAME='United States',ORIGIN_COUNTRY_NAME='Romania', count=35)]))

In [11]:
# dataframe tranformation
# Transformations in Spark DataFrames are lazy operations that do not execute immediately but create a logical plan.
df.select("ORIGIN_COUNTRY_NAME").show(6)

+-------------------+
|ORIGIN_COUNTRY_NAME|
+-------------------+
|            Romania|
|            Croatia|
|            Ireland|
|      United States|
|              India|
|          Singapore|
+-------------------+
only showing top 6 rows



In [12]:
# creating dataframes
data = [(1, 'a'), (2, 'b')]
columns = ['id', 'letter']
df1 = spark.createDataFrame(data, columns)
df1.show()

+---+------+
| id|letter|
+---+------+
|  1|     a|
|  2|     b|
+---+------+



In [13]:
# select
from pyspark.sql.functions import expr
df.select(expr("DEST_COUNTRY_NAME AS destination").alias("DEST_COUNTRY_NAME")).show(3)

+-----------------+
|DEST_COUNTRY_NAME|
+-----------------+
|    United States|
|    United States|
|    United States|
+-----------------+
only showing top 3 rows



In [14]:
# selectExpr
df.selectExpr("DEST_COUNTRY_NAME AS destination", "DEST_COUNTRY_NAME").show(2)

+-------------+-----------------+
|  destination|DEST_COUNTRY_NAME|
+-------------+-----------------+
|United States|    United States|
|United States|    United States|
+-------------+-----------------+
only showing top 2 rows



In [15]:
# adding columns
from pyspark.sql.functions import lit
df.select(expr("*"), lit(1).alias("One")).show()

+--------------------+-------------------+-----+---+
|   DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|One|
+--------------------+-------------------+-----+---+
|       United States|            Romania|   15|  1|
|       United States|            Croatia|    1|  1|
|       United States|            Ireland|  344|  1|
|               Egypt|      United States|   15|  1|
|       United States|              India|   62|  1|
|       United States|          Singapore|    1|  1|
|       United States|            Grenada|   62|  1|
|          Costa Rica|      United States|  588|  1|
|             Senegal|      United States|   40|  1|
|             Moldova|      United States|    1|  1|
|       United States|       Sint Maarten|  325|  1|
|       United States|   Marshall Islands|   39|  1|
|              Guyana|      United States|   64|  1|
|               Malta|      United States|    1|  1|
|            Anguilla|      United States|   41|  1|
|             Bolivia|      United States|   3

In [16]:
# renaming columns
df.withColumnRenamed("DEST_COUNTRY_NAME", "dest country").show(2)

+-------------+-------------------+-----+
| dest country|ORIGIN_COUNTRY_NAME|count|
+-------------+-------------------+-----+
|United States|            Romania|   15|
|United States|            Croatia|    1|
+-------------+-------------------+-----+
only showing top 2 rows



In [18]:
# changing column types
df.withColumn("count", col("count").cast("long")).select("count").show()

+-----+
|count|
+-----+
|   15|
|    1|
|  344|
|   15|
|   62|
|    1|
|   62|
|  588|
|   40|
|    1|
|  325|
|   39|
|   64|
|    1|
|   41|
|   30|
|    6|
|    4|
|  230|
|    1|
+-----+
only showing top 20 rows



In [19]:
# filtering rows
df.where("count < 30").show(5)
df.filter(col("count") > 30).show(5)

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Romania|   15|
|    United States|            Croatia|    1|
|            Egypt|      United States|   15|
|    United States|          Singapore|    1|
|          Moldova|      United States|    1|
+-----------------+-------------------+-----+
only showing top 5 rows

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Ireland|  344|
|    United States|              India|   62|
|    United States|            Grenada|   62|
|       Costa Rica|      United States|  588|
|          Senegal|      United States|   40|
+-----------------+-------------------+-----+
only showing top 5 rows



In [20]:
# getting unique rows
df.select("ORIGIN_COUNTRY_NAME", "DEST_COUNTRY_NAME").distinct().count()

256

In [21]:
# sorting
from pyspark.sql.functions import asc, desc
df.sort("count").show(5)
df.orderBy("count", "ORIGIN_COUNTRY_NAME").show(5)
df.orderBy(col("count").asc(), col("ORIGIN_COUNTRY_NAME").desc()).show(5)

+--------------------+-------------------+-----+
|   DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+--------------------+-------------------+-----+
|               Malta|      United States|    1|
|Saint Vincent and...|      United States|    1|
|       United States|            Croatia|    1|
|       United States|          Gibraltar|    1|
|       United States|          Singapore|    1|
+--------------------+-------------------+-----+
only showing top 5 rows

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Bahrain|    1|
|    United States|           Bulgaria|    1|
|    United States|            Croatia|    1|
|    United States|             Cyprus|    1|
|    United States|            Estonia|    1|
+-----------------+-------------------+-----+
only showing top 5 rows

+--------------------+-------------------+-----+
|   DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|coun

In [22]:
df.limit(8).show()

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Romania|   15|
|    United States|            Croatia|    1|
|    United States|            Ireland|  344|
|            Egypt|      United States|   15|
|    United States|              India|   62|
|    United States|          Singapore|    1|
|    United States|            Grenada|   62|
|       Costa Rica|      United States|  588|
+-----------------+-------------------+-----+



In [ ]:
# repartion
df_semi_partition_5 = dfm.repartition(5, col("DEST_COUNTRY_NAME"))
df_semi_partition_5.rdd.getNumPartitions()

5

In [23]:
# coalesce
df_coalesce = df.repartition(6, col("DEST_COUNTRY_NAME")).coalesce(2)
df_coalesce.rdd.getNumPartitions()

2

Day 3 (chapter 7)
1. Aggregation Functions
        1. count
        2. countDistinct
        3. first and last
        4. min and max
        5. sum
        6. sum
        7. sumDistinct
        8. avg
        9. grouping
        10. window functions

In [24]:
# count
from pyspark.sql.functions import count
df.select(count("ORIGIN_COUNTRY_NAME")).show()

+--------------------------+
|count(ORIGIN_COUNTRY_NAME)|
+--------------------------+
|                       257|
+--------------------------+



In [25]:
# countDistinct
from pyspark.sql.functions import countDistinct
df.select(countDistinct("ORIGIN_COUNTRY_NAME")).show()

+-----------------------------------+
|count(DISTINCT ORIGIN_COUNTRY_NAME)|
+-----------------------------------+
|                                125|
+-----------------------------------+



In [26]:
# first and last
from pyspark.sql.functions import first, last
df.select(first("ORIGIN_COUNTRY_NAME"), last("count")).show()

+--------------------------+-----------+
|first(ORIGIN_COUNTRY_NAME)|last(count)|
+--------------------------+-----------+
|                   Romania|         35|
+--------------------------+-----------+



In [27]:
# min and max
from pyspark.sql.functions import min, max
df.select(min("count"), max("count")).show()


+----------+----------+
|min(count)|max(count)|
+----------+----------+
|         1|    370002|
+----------+----------+



In [28]:
# sum
from pyspark.sql.functions import sum
df.select(sum("count")).show()

+----------+
|sum(count)|
+----------+
|    453351|
+----------+



In [29]:
# sumDistinct
from pyspark.sql.functions import sumDistinct
df.select(sumDistinct("count")).show()

/usr/local/lib/python3.11/dist-packages/pyspark/sql/functions.py:988: FutureWarning: Deprecated in 3.2, use sum_distinct instead.
  warnings.warn("Deprecated in 3.2, use sum_distinct instead.", FutureWarning)


+-------------------+
|sum(DISTINCT count)|
+-------------------+
|             450718|
+-------------------+



In [30]:
# avg
from pyspark.sql.functions import avg
df.select(avg("count")).show()

+-----------------+
|       avg(count)|
+-----------------+
|1764.011673151751|
+-----------------+



In [31]:
# grouping
df.groupBy("Dest_COUNTRY_NAME").count().show(5)

+-----------------+-----+
|Dest_COUNTRY_NAME|count|
+-----------------+-----+
|         Anguilla|    1|
|           Russia|    1|
|         Paraguay|    1|
|          Senegal|    1|
|           Sweden|    1|
+-----------------+-----+
only showing top 5 rows



In [33]:
# Window functions
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, rank, dense_rank, lag, lead

windowSpec = Window.partitionBy("ORIGIN_COUNTRY_NAME").orderBy(col("count").desc())

df.withColumn("row_number", row_number().over(windowSpec)) \
              .withColumn("rank", rank().over(windowSpec)) \
              .withColumn("dense_rank", dense_rank().over(windowSpec)).show(6)

+-----------------+-------------------+-----+----------+----+----------+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|row_number|rank|dense_rank|
+-----------------+-------------------+-----+----------+----+----------+
|    United States|             Angola|   13|         1|   1|         1|
|    United States|           Anguilla|   38|         1|   1|         1|
|    United States|Antigua and Barbuda|  117|         1|   1|         1|
|    United States|          Argentina|  141|         1|   1|         1|
|    United States|              Aruba|  342|         1|   1|         1|
|    United States|          Australia|  258|         1|   1|         1|
+-----------------+-------------------+-----+----------+----+----------+
only showing top 6 rows



Day 4 (chapter 8)
1. Joins
        1. Try out examples for each types of joins
        2. Handling Duplicate column names
        3. How spark performs joins

2. Handling Duplicate column names
---- a. the easiest fix is to change the join expression from a Boolean expression to a string or sequence
b. Another approach is to drop the offending column after the join
c. rename one of our columns before the join

3. How spark performs joins
---- To understand how Spark performs joins, you need to understand the two core resources at play: the node-to-node communication strategy and per node computation strategy

a. In Spark, data is distributed across multiple nodes (machines) in a cluster. When performing a join between two DataFrames (or RDDs), Spark needs to ensure that data is correctly shuffled between these nodes to facilitate the join operation. Here's how this communication works:

Shuffling:
Shuffling refers to the process of redistributing data across partitions in the cluster. When performing joins, Spark may need to shuffle data between different nodes, depending on the type of join operation and the data distribution. This is necessary to ensure that all matching keys (e.g., rows with the same value in a column) end up on the same node for the join.
This shuffle is a costly operation because it involves disk and network IO, leading to delays in the execution of the join.

Broadcasting:
In some cases, Spark uses broadcast joins. If one of the DataFrames (or tables) is small enough to fit into the memory of each node, Spark can broadcast this small DataFrame to all the nodes in the cluster. This eliminates the need for shuffling since each node can perform the join operation locally.

b. Per-Node Computation Strategy
Each node in the cluster executes the join operation on its local data, but the actual strategy may differ based on the type of join. Spark has different computation strategies that allow it to execute joins efficiently across all nodes in parallel

In [34]:
person = spark.createDataFrame([
(0, "Bill Chambers", 0, [100]),
(1, "Matei Zaharia", 1, [500, 250, 100]),
(2, "Michael Armbrust", 1, [250, 100])])\
.toDF("id", "name", "graduate_program", "spark_status")

graduateProgram = spark.createDataFrame([
(0, "Masters", "School of Information", "UC Berkeley"),
(2, "Masters", "EECS", "UC Berkeley"),
(1, "Ph.D.", "EECS", "UC Berkeley")])\
.toDF("id", "degree", "department", "school")

sparkStatus = spark.createDataFrame([
(500, "Vice President"),
(250, "PMC Member"),
(100, "Contributor")])\
.toDF("id", "status")

person.createOrReplaceTempView("person")
graduateProgram.createOrReplaceTempView("graduateProgram")
sparkStatus.createOrReplaceTempView("sparkStatus")

In [35]:
person.show()
graduateProgram.show()
sparkStatus.show()

+---+----------------+----------------+---------------+
| id|            name|graduate_program|   spark_status|
+---+----------------+----------------+---------------+
|  0|   Bill Chambers|               0|          [100]|
|  1|   Matei Zaharia|               1|[500, 250, 100]|
|  2|Michael Armbrust|               1|     [250, 100]|
+---+----------------+----------------+---------------+

+---+-------+--------------------+-----------+
| id| degree|          department|     school|
+---+-------+--------------------+-----------+
|  0|Masters|School of Informa...|UC Berkeley|
|  2|Masters|                EECS|UC Berkeley|
|  1|  Ph.D.|                EECS|UC Berkeley|
+---+-------+--------------------+-----------+

+---+--------------+
| id|        status|
+---+--------------+
|500|Vice President|
|250|    PMC Member|
|100|   Contributor|
+---+--------------+



In [36]:
# inner
joinExpression = person["graduate_program"] == graduateProgram['id']
person.join(graduateProgram, joinExpression, "inner").show()

+---+----------------+----------------+---------------+---+-------+--------------------+-----------+
| id|            name|graduate_program|   spark_status| id| degree|          department|     school|
+---+----------------+----------------+---------------+---+-------+--------------------+-----------+
|  0|   Bill Chambers|               0|          [100]|  0|Masters|School of Informa...|UC Berkeley|
|  1|   Matei Zaharia|               1|[500, 250, 100]|  1|  Ph.D.|                EECS|UC Berkeley|
|  2|Michael Armbrust|               1|     [250, 100]|  1|  Ph.D.|                EECS|UC Berkeley|
+---+----------------+----------------+---------------+---+-------+--------------------+-----------+



In [37]:
# outer
person.join(graduateProgram, joinExpression, "outer").show()

+----+----------------+----------------+---------------+---+-------+--------------------+-----------+
|  id|            name|graduate_program|   spark_status| id| degree|          department|     school|
+----+----------------+----------------+---------------+---+-------+--------------------+-----------+
|   0|   Bill Chambers|               0|          [100]|  0|Masters|School of Informa...|UC Berkeley|
|   1|   Matei Zaharia|               1|[500, 250, 100]|  1|  Ph.D.|                EECS|UC Berkeley|
|   2|Michael Armbrust|               1|     [250, 100]|  1|  Ph.D.|                EECS|UC Berkeley|
|NULL|            NULL|            NULL|           NULL|  2|Masters|                EECS|UC Berkeley|
+----+----------------+----------------+---------------+---+-------+--------------------+-----------+



In [38]:
# left
person.join(graduateProgram, joinExpression, "left").show()

+---+----------------+----------------+---------------+---+-------+--------------------+-----------+
| id|            name|graduate_program|   spark_status| id| degree|          department|     school|
+---+----------------+----------------+---------------+---+-------+--------------------+-----------+
|  0|   Bill Chambers|               0|          [100]|  0|Masters|School of Informa...|UC Berkeley|
|  1|   Matei Zaharia|               1|[500, 250, 100]|  1|  Ph.D.|                EECS|UC Berkeley|
|  2|Michael Armbrust|               1|     [250, 100]|  1|  Ph.D.|                EECS|UC Berkeley|
+---+----------------+----------------+---------------+---+-------+--------------------+-----------+



In [39]:
# right
person.join(graduateProgram, joinExpression, "right").show()

+----+----------------+----------------+---------------+---+-------+--------------------+-----------+
|  id|            name|graduate_program|   spark_status| id| degree|          department|     school|
+----+----------------+----------------+---------------+---+-------+--------------------+-----------+
|   0|   Bill Chambers|               0|          [100]|  0|Masters|School of Informa...|UC Berkeley|
|   2|Michael Armbrust|               1|     [250, 100]|  1|  Ph.D.|                EECS|UC Berkeley|
|   1|   Matei Zaharia|               1|[500, 250, 100]|  1|  Ph.D.|                EECS|UC Berkeley|
|NULL|            NULL|            NULL|           NULL|  2|Masters|                EECS|UC Berkeley|
+----+----------------+----------------+---------------+---+-------+--------------------+-----------+



In [40]:
# left_semi
person.join(graduateProgram, joinExpression, "left_semi").show()

+---+----------------+----------------+---------------+
| id|            name|graduate_program|   spark_status|
+---+----------------+----------------+---------------+
|  0|   Bill Chambers|               0|          [100]|
|  1|   Matei Zaharia|               1|[500, 250, 100]|
|  2|Michael Armbrust|               1|     [250, 100]|
+---+----------------+----------------+---------------+



In [41]:
# left_anti
person.join(graduateProgram, joinExpression, "left_anti").show()

+---+----+----------------+------------+
| id|name|graduate_program|spark_status|
+---+----+----------------+------------+
+---+----+----------------+------------+



In [42]:
# cross
person.join(graduateProgram, joinExpression, "cross").show()

+---+----------------+----------------+---------------+---+-------+--------------------+-----------+
| id|            name|graduate_program|   spark_status| id| degree|          department|     school|
+---+----------------+----------------+---------------+---+-------+--------------------+-----------+
|  0|   Bill Chambers|               0|          [100]|  0|Masters|School of Informa...|UC Berkeley|
|  1|   Matei Zaharia|               1|[500, 250, 100]|  1|  Ph.D.|                EECS|UC Berkeley|
|  2|Michael Armbrust|               1|     [250, 100]|  1|  Ph.D.|                EECS|UC Berkeley|
+---+----------------+----------------+---------------+---+-------+--------------------+-----------+



Day 5 (chapter 9)
1. Datasources
        1. Basics of reading data
        ---- "spark.read" used to read the data sources.
             After we have a DataFrame reader, we specify several values: The format, The schema, The read mode, A series of options.
             Read modes: Reading data from an external source naturally entails encountering malformed data, especially when working with only semi-structured data sources. Read modes specify what will happen when Spark does come across malformed records.
             Read mode Description:
             permissive --- Sets all fields to null when it encounters a corrupted record and places all corrupted records in a string column called _corrupt_record.
             dropMalformed --- Drops the row that contains malformed records.
             failFast --- Fails immediately upon encountering malformed records The default is permissive

        2. Basics of write data
        ---- "DataFrameWriter.format(...).option(...).partitionBy(...).bucketBy(...).sortBy(...).save()" used to save in data sources.
             Save modes specify what will happen if Spark finds data at the specified location (assuming all else equal).
             Save mode Description:
             append ---  Appends the output files to the list of files that already exist at that location.
             overwrite --- Will completely overwrite any data that already exists there.
             errorIfExists ---- Throws an error and fails the write if data or files already exist at the specified location.
             ignore --- If data or files exist at the location, do nothing with the current DataFrame.

        3. CSV files - reading, writing
        ---- CSV stands for commma-separated values. This is a common text file format in which each line represents a single record, and commas separate each field within a record.
        "spark.read.format("csv")" used to read from csv files.
        "csvFile = spark.read.format("csv").option("header", "true").option("mode", "FAILFAST").option("inferSchema", "true").load("/data/flight-data/csv/2010-summary.csv")" used to write in csv files.

        4. REading and writing json files
        ---- "spark.read.format("json").option("mode", "FAILFAST").option("inferSchema", "true").load("/data/flight-data/json/2010-summary.json").show(5)" used to read from json files.
        "csvFile.write.format("json").mode("overwrite").save("/tmp/my-json-file.json")" used to write in json files.

        5. Parquet files - important
        ---- Parquet is an open source column-oriented data store that provides a variety of storage optimizations, especially for analytics workloads. It provides columnar compression, which saves storage space and allows for reading individual columns instead of entire files. It is a file format that works exceptionally well with Apache Spark and is in fact the default file format. We recommend writing data out to Parquet for long-term storage because reading from a Parquet file will always be more efficient than JSON or CSV. Another advantage of Parquet is that it supports complex types. Parquet data source options: compression or codec, mergeSchema

        6. Reading and Writing parquet files
        ---- "spark.read.format("parquet").load("/data/flight-data/parquet/2010-summary.parquet").show(5)" used to read from parquet files.
             "csvFile.write.format("parquet").mode("overwrite").save("/tmp/my-parquet-file.parquet")" used to write in parquet files.

        7. orc - optional
        ---- ORC is a self-describing, type-aware columnar file format designed for Hadoop workloads. It is optimized for large streaming reads, but with integrated support for finding required rows quickly. the fundamental difference is that Parquet is further optimized for use with Spark, whereas ORC is further optimized for Hive.

        8. Splittable File Types and COmpression
        ---- Certain file formats are fundamentally “splittable.” This can improve speed because it makes it possible for Spark to avoid reading an entire file, and access only the parts of the file necessary to satisfy your query. Additionally if you’re using something like Hadoop Distributed File System (HDFS), splitting a file can provide further optimization if that file spans multiple blocks. Not all compression schemes are splittable. How you store your data is of immense consequence when it comes to making your Spark jobs run smoothly. We recommend Parquet with gzip compression.

        9. Managing File size
        ---- Managing file sizes is an important factor not so much for writing data but reading it later on. When you’re writing lots of small files, there’s a significant metadata overhead that you incur managing all of those files. you don’t want files that are too large either, because it becomes inefficient to have to read entire blocks of data when you need only a few rows. You can use the maxRecordsPerFile option and specify a number of your choosing. This allows you to better control file sizes by controlling the number of records that are written to each file.